# 04 Infer Cluster ADS
Bestes Modell laden, ADS blockweise scoren, DBSCAN-Clustering, Exporte.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = "replace_with_run_id_from_00"
DEVICE = "auto"
import json
import numpy as np
import pandas as pd


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
from src.common.config import load_notebook_context, build_run_dirs
from src.common.io_schema import read_parquet
from src.approaches.nand.infer_pairs import score_pairs_with_checkpoint
from src.approaches.nand.cluster import cluster_blockwise_dbscan
from src.approaches.nand.export import build_publication_author_mapping

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
CLUSTER_CFG = CTX["CLUSTER"]

RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "checkpoints", "pair_scores", "clusters", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
pair_score_dir = RUN_DIRS["pair_scores"]
cluster_dir = RUN_DIRS["clusters"]
metrics_dir = RUN_DIRS["metrics"]

ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")
ads_pairs = read_parquet(subset_dir / f"ads_pairs_{RUN_STAGE}.parquet")
ads_chars = np.load(emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy")
ads_text = np.load(emb_dir / f"ads_specter_{RUN_STAGE}.npy")

with (metrics_dir / "03_train_manifest.json").open("r", encoding="utf-8") as f:
    train_manifest = json.load(f)

best_checkpoint = train_manifest["best_checkpoint"]
best_threshold = train_manifest["best_threshold"]
print("Best checkpoint:", best_checkpoint)
print("Best threshold:", best_threshold)


Best checkpoint: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\checkpoints\replace_with_run_id_from_00\replace_with_run_id_from_00_seed1.pt
Best threshold: -1.0


In [3]:
pair_scores_path = pair_score_dir / f"ads_pair_scores_{RUN_STAGE}.parquet"
pair_scores = score_pairs_with_checkpoint(
    mentions=ads_mentions,
    pairs=ads_pairs,
    chars2vec=ads_chars,
    text_emb=ads_text,
    checkpoint_path=best_checkpoint,
    output_path=pair_scores_path,
    device=DEVICE,
)
print("Pair scores:", len(pair_scores), "->", pair_scores_path)


Pair scores: 500 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\pair_scores\replace_with_run_id_from_00\ads_pair_scores_smoke.parquet


C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\src\approaches\nand\infer_pairs.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fea

In [4]:
clusters_path = cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet"
clusters = cluster_blockwise_dbscan(
    mentions=ads_mentions,
    pair_scores=pair_scores,
    cluster_config=CLUSTER_CFG,
    output_path=clusters_path,
)
print("Clusters:", len(clusters), "->", clusters_path)


Clusters: 1000 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\clusters\replace_with_run_id_from_00\ads_clusters_smoke.parquet


In [5]:
merged = ads_mentions[["mention_id", "block_key"]].merge(clusters, on=["mention_id", "block_key"], how="left")
cluster_size = merged.groupby(["block_key", "author_uid"]).size().rename("size").reset_index()

singleton_ratio = float((cluster_size["size"] == 1).mean()) if len(cluster_size) else 0.0
print("Singleton ratio:", singleton_ratio)
print("Cluster size describe:")
print(cluster_size["size"].describe())

cluster_size.groupby("block_key")["size"].agg(["count", "max"]).sort_values(["count","max"], ascending=False).head(20)


Singleton ratio: 0.8372093023255814
Cluster size describe:
count    860.000000
mean       1.162791
std        0.369389
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        2.000000
Name: size, dtype: float64


,count,max
block_key,,
a.boksenberg,2,1
a.cameron,2,1
a.dalgarno,2,1
a.filippenko,2,1
a.green,2,1
a.harris,2,1
a.martin,2,1
a.moffat,2,1
a.sen,2,1


In [6]:
mention_export = cluster_dir / f"mention_author_uid_{RUN_STAGE}.parquet"
pub_export = cluster_dir / f"publication_authors_{RUN_STAGE}.parquet"

clusters.to_parquet(mention_export, index=False)
pub_map = build_publication_author_mapping(mentions=ads_mentions, clusters=clusters, output_path=pub_export)

print("mention export:", mention_export)
print("publication export:", pub_export)
print("publication rows:", len(pub_map))


mention export: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\clusters\replace_with_run_id_from_00\mention_author_uid_smoke.parquet
publication export: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\clusters\replace_with_run_id_from_00\publication_authors_smoke.parquet
publication rows: 1000
